# Position Assigner — Model Training

Trains a classifier to label each telemetry row with one of 11 fine-grained
flight attitude classes:

| Label | Meaning |
|---|---|
| `taxing` | On the ground, taxiing |
| `landing-takeoff` | Takeoff or landing roll |
| `ascending_straight` | Climbing, wings level |
| `ascending_left` | Climbing, left bank |
| `ascending_right` | Climbing, right bank |
| `stable_straight` | Level flight, wings level |
| `stable_left` | Level flight, left bank |
| `stable_right` | Level flight, right bank |
| `descending_straight` | Descending, wings level |
| `descending_left` | Descending, left bank |
| `descending_right` | Descending, right bank |

**Features used:** `acc_x`, `pitch`, `roll`, `turn_rate`, `magnetic_heading`, `ground_speed`, `vertical_speed`

**Config:** Place a `config.yml` inside the `notebooks/` folder with the shape:
```yaml
database:
  username: your-db-user
  password: your-db-password
  ip_address: your-db-host
  port: '50000'
  db_name: YOUR_DB_NAME
```

In [ ]:
import sys
sys.path.insert(0, '..')  # so we can import from the project root

import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

from db_connection_params_handler import ConnectionParamsHandler
from db_factory import make_db_manager
from db_handlers.ibm_db2.MQTT_db_access import TelemetryMQTTDatabaseAccess
from mappers.telemetry_mapper import FormattedTelemetryMapper
from logic.position_assigner_instance import position_assigner

## 1. Connect to the database

In [ ]:
CONFIG_PATH = 'config.yml'

connection = ConnectionParamsHandler(connection_filename=CONFIG_PATH)
db_manager = make_db_manager(connection)
telemetry_access = TelemetryMQTTDatabaseAccess(manager=db_manager)
print('Connected.')

## 2. Pull telemetry with current attitude labels

We run the existing `PositionAssigner` pipeline to produce labelled rows,
then treat its `attitude` output as the ground truth for retraining.
You can also provide manually corrected labels by editing the DataFrame below.

In [ ]:
QUERIES = [
    # (registration, start_date, end_date)
    ('I-PVLG', '2024-01-01', '2024-06-30'),
    # add more as needed
]

frames = []
for marca, start, end in QUERIES:
    print(f'Fetching {marca}  {start} → {end} ...')
    raw = telemetry_access.get_telemetry_by_marca_datetime_interval(
        marca=marca, start_datetime=start, end_datetime=end
    )
    if raw.dropna(how='all').empty:
        print('  no data')
        continue
    mapper = FormattedTelemetryMapper(df_chunk_generator=raw, position_assigner=position_assigner)
    df = mapper.new_generate_telemetry_samples()
    frames.append(df)
    print(f'  {len(df):,} rows')

data = pd.concat(frames, ignore_index=True)
print(f'\nTotal rows: {len(data):,}')
data.head()

## 3. Prepare features and labels

In [ ]:
FEATURES = ['acc_x', 'pitch', 'roll', 'turn_rate', 'magnetic_heading', 'ground_speed', 'vertical_speed']

# Same clipping as FlightSeparator (catch sensor outliers)
data['pitch']          = data['pitch'].clip(-180, 180)
data['roll']           = data['roll'].clip(-180, 180)
data['ground_speed']   = data['ground_speed'].clip(-1, 500)
data['vertical_speed'] = data['vertical_speed'].clip(-5000, 5000)

# Drop rows where any feature or label is missing
data = data.dropna(subset=FEATURES + ['attitude'])

print(data['attitude'].value_counts())
data[FEATURES + ['attitude']].head()

## 4. Train / test split

In [ ]:
X = data[FEATURES].values
y = data['attitude'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train):,}   Test: {len(X_test):,}')

## 5. Train model

The existing model is a Random Forest (`f1.pkl`). You can swap in any
sklearn-compatible classifier without changing any project code.

In [ ]:
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=3,
    n_jobs=-1,
    random_state=42,
)
model.fit(X_train, y_train)
print('Training complete.')

## 6. Evaluate

In [ ]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

cv_scores = cross_val_score(model, X, y, cv=5, scoring='f1_weighted', n_jobs=-1)
print(f'5-fold CV F1 (weighted): {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)
disp.plot(ax=ax, cmap='Blues', xticks_rotation=45)
plt.title('Position Assigner — Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
importances.plot.barh()
plt.title('Feature importances')
plt.tight_layout()
plt.show()

## 7. Save artefacts

Drop the output files into `logic/position_assignment_data/` to replace `f1.pkl`.

> **Note on label format:** `PositionAssigner` loads `label_names` as a
> `{class_name: integer_index}` dict (inverse of the model's class ordering).
> The cell below builds that mapping automatically from `model.classes_`.

In [ ]:
OUT_DIR = Path('../logic/position_assignment_data')
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = 'retrained'  # change to 'f1' to replace the existing model in-place

# Model
joblib.dump(model, OUT_DIR / f'{MODEL_NAME}.pkl')

# Class names — {class_name: index} as expected by PositionAssigner
class_names = {name: idx for idx, name in enumerate(model.classes_)}
with open(OUT_DIR / f'{MODEL_NAME}_class_names.json', 'w') as f:
    json.dump(class_names, f, indent=2)

# Feature list
with open(OUT_DIR / f'{MODEL_NAME}_selected_features.json', 'w') as f:
    json.dump(FEATURES, f, indent=2)

print('Saved:')
for p in OUT_DIR.glob(f'{MODEL_NAME}*'):
    print(' ', p)

print('\nTo use the new model, update logic/position_assigner_instance.py:')
print(f"  model_file_path='logic/position_assignment_data/{MODEL_NAME}.pkl'")
print(f"  label_names_file_path='logic/position_assignment_data/{MODEL_NAME}_class_names.json'")
print(f"  features_file_path='logic/position_assignment_data/{MODEL_NAME}_selected_features.json'")